In [ ]:
# Import libraries
import pandas as pd
import os
from pathlib import Path
import networkx as nx
import numpy as np
import pickle

from utils.graph import create_graph, draw_graph, jaccard_similarity

In [ ]:
timeseries_dir= r"data/MIMICIII_last48h_ts2h/timeseries"
train_dir = r"data/MIMICIII_last48h_ts2h/train"
Path(train_dir).mkdir(exist_ok=True, parents=True)

## Build Drug Graph

In the comorbidity graph G_d=(N_d, E_d, r_d), the nodes N_d are disease sepcified by icd9 codes, the edges E_d are the comorbidity of diseases in the same admission, and the relationship r_d is the drug similarity.

In [ ]:
# load data
demo_df = pd.read_csv(os.path.join(train_dir, "demographics.csv"), sep=',', dtype={'icd9_code': str})
icd9_codes = demo_df["icd9_code"].unique()
hadm_ids = demo_df["hadm_id"].unique()

drug_df = pd.read_csv(os.path.join(timeseries_dir, "drug.csv"), sep=',', dtype={'icd9_code': str})
drug_df = drug_df[drug_df['icd9_code'].isin(icd9_codes)]
drug_df = drug_df[drug_df['hadm_id'].isin(hadm_ids)]

drug_df.info()

In [ ]:
# create drug usage graph
# 创建药物使用图
G_d = nx.Graph()

# 为每个节点添加属性：存储对应的 drug_sequence
for icd9_code in drug_df.icd9_code.unique():
    drug_sequence = drug_df[drug_df['icd9_code'] == icd9_code]['drug'].unique()
    G_d.add_node(icd9_code, drug_sequence=drug_sequence)

# 计算边的权重（基于 Jaccard 相似度）
for i, icd9_code1 in enumerate(G_d.nodes()):
    drug_sequence1 = G_d.nodes[icd9_code1]['drug_sequence']
    for icd9_code2 in list(G_d.nodes())[i+1:]:
        drug_sequence2 = G_d.nodes[icd9_code2]['drug_sequence']
        if len(drug_sequence1) == 0 or len(drug_sequence2) == 0:
            similarity = 0
        else:
            similarity = jaccard_similarity(drug_sequence1, drug_sequence2)
        G_d.add_edge(icd9_code1, icd9_code2, weight=similarity)

# delete nodes' attributes
for node in G_d.nodes:
    if 'drug_sequence' in G_d.nodes[node]:
        del G_d.nodes[node]['drug_sequence']

# 进行 Min-Max 归一化
max_weight = max([d['weight'] for _, _, d in G_d.edges(data=True)])
min_weight = min([d['weight'] for _, _, d in G_d.edges(data=True)])
for u, v, d in G_d.edges(data=True):
    G_d[u][v]['weight'] = (G_d[u][v]['weight'] - min_weight) / (max_weight - min_weight)

# 保留权重最高的 50% 的边
remove_threshold = np.percentile([d['weight'] for _, _, d in G_d.edges(data=True)], 50)
edges_to_remove = [(u, v) for u, v, d in G_d.edges(data=True) if d['weight'] < remove_threshold]
G_d.remove_edges_from(edges_to_remove)

# 保存图
with open(os.path.join(train_dir, "drug_graph.pkl"), 'wb') as f:
    pickle.dump(G_d, f)

In [ ]:
with open(os.path.join(train_dir, "drug_graph.pkl"), "rb") as f:
    G_d_target = pickle.load(f)

draw_graph(G_d_target, node_num=20)